In [1]:
import os
import time
from pathlib import Path

import earthaccess
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj

d:\conda\envs\mnf-ningaloo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_latlon_xr(ds):
    """Calculate latitude and longitude from a geostationary dataset."""
    # ONLY NEEDED FOR L2P DATASET

    x = ds['ni'].values
    y = ds['nj'].values

    geo_var = ds['geostationary']

    lon_0 = float(geo_var.longitude_of_projection_origin)
    h     = float(geo_var.perspective_point_height)
    sweep = str(geo_var.sweep_angle_axis)  

    p = Proj(proj='geos', h=h, lon_0=lon_0, sweep=sweep, datum='WGS84')

    X, Y = np.meshgrid(x*h, y*h)
    lon, lat = p(X, Y, inverse=True)

    # fill space pixels with NANs
    lon[np.abs(lon) > 360.0] = np.nan
    lat[np.abs(lat) > 90.0]  = np.nan

    return X, Y, lat, lon, x, y

In [3]:
def create_coords_dataset_xr(ds):
    """Create an xarray dataset with coordinates for a geostationary dataset."""
    # ONLY NEEDED FOR L2P DATASET
    
    X, Y, lat, lon, x, y = calculate_latlon_xr(ds)

    # create xarray dataset
    ds_coords = xr.Dataset(
        coords={
            'ni': ('ni', x),
            'nj': ('nj', y),
            'lat': (('nj','ni'), lat),
            'lon': (('nj','ni'), lon),
            'X': (('nj','ni'), X),
            'Y': (('nj','ni'), Y),
        }
    )

    # assign attributes
    ds_coords['lat'].attrs.update(long_name='latitude', units='degrees_north')
    ds_coords['lon'].attrs.update(long_name='longitude', units='degrees_east')
    ds_coords['X'].attrs.update(long_name='geostationary X', units='m')
    ds_coords['Y'].attrs.update(long_name='geostationary Y', units='m')

    return ds_coords


In [4]:
def get_ilims_jlims(lon, lat, lonlims, latlims):
    """Get the i and j limits for cropping based on longitude and latitude limits."""
    # OPTIONAL FOR L2P DATASET, NOT NEEDED FOR L3C DATASET    
    
    def find_nearest(x_grid, y_grid, x_point, y_point):
        """Find the (j, i) indices in x_grid, y_grid closest to the point (x_point, y_point), ignoring NaNs."""
        # Create a mask for valid points
        valid_mask = ~np.isnan(x_grid) & ~np.isnan(y_grid)
        
        # Compute distances only where valid
        
        distances = np.full_like(x_grid, np.inf, dtype=float)
        distances[valid_mask] = np.hypot(x_grid[valid_mask] - x_point,
                                        y_grid[valid_mask] - y_point)
        
        return np.unravel_index(np.argmin(distances), distances.shape)


    # # Find grid indices of the four corner points
    j1, i1 = find_nearest(lon, lat, lonlims[0], latlims[0])
    j2, i2 = find_nearest(lon, lat, lonlims[1], latlims[0])
    j3, i3 = find_nearest(lon, lat, lonlims[0], latlims[1])
    j4, i4 = find_nearest(lon, lat, lonlims[1], latlims[1])
    
    print(f"Indices: ({i1}, {j1}), ({i2}, {j2}), ({i3}, {j3}), ({i4}, {j4})")
    
    ilims = (min(i1, i2, i3, i4), max(i1, i2, i3, i4) + 1)
    jlims = (min(j1, j2, j3, j4), max(j1, j2, j3, j4) + 1)

    return ilims, jlims

In [5]:
def retry(func, retries=20, wait_seconds=60, stop_if_error=True, *args, **kwargs):
    """Retry a function call with a specified number of retries and wait time."""
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"[Retry] Attempt {attempt}/{retries} failed: {e}")
            time.sleep(wait_seconds)
            
    if stop_if_error:
        raise RuntimeError(f"Function '{func.__name__}' failed after {retries} retries.")
    else: 
        print(f"[Retry] Function '{func.__name__}' failed after {retries} retries, but continuing execution.")


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import earthaccess

# 常量：Himawari-9 L3C (ACSPO v2.90)
COLLECTION_SHORT_NAME = "H09-AHI-L3C-ACSPO-v2.90"
FILENAME_TIME_RE = re.compile(r"(\d{14})-STAR-L3C_")

def ensure_earthdata_login(netrc_path: Path = Path(".netrc")):
    """使用 .netrc 登录 Earthdata"""
    if not netrc_path.exists():
        raise FileNotFoundError(
            f".netrc not found at {netrc_path}\n"
            "machine urs.earthdata.nasa.gov\n  login YOUR_USERNAME\n  password YOUR_PASSWORD\n"
        )
    os.environ["NETRC"] = str(netrc_path.resolve())
    # 避免代理干扰
    for k in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
        os.environ.pop(k, None)
    auth = earthaccess.login(strategy="netrc", persist=True)
    if not auth.authenticated:
        raise RuntimeError("Earthdata login failed. Check .netrc")

def _cap_temporal_by_delay(t0: str, t1: str, delay_hours: int = 4):
    """限制结束时间，避免近实时数据尚未发布"""
    end = np.datetime64(t1)
    now_safe = np.datetime64(pd.Timestamp.utcnow().to_pydatetime()) - np.timedelta64(delay_hours, "h")
    if end > now_safe:
        end = now_safe
    return t0, str(end.astype("datetime64[s]"))

def _bbox(lonlims, latlims):
    """返回 bounding_box 格式 (W,S,E,N)"""
    west, east = min(lonlims), max(lonlims)
    south, north = min(latlims), max(latlims)
    return (west, south, east, north)

def query_himawari_l3c_manifest(
    timelims,
    lonlims,
    latlims,
    delay_hours=4,
    max_results=None,
    netrc_path=Path(".netrc"),
):
    """
    查询 Himawari-9 L3C 可用数据清单
    """
    ensure_earthdata_login(netrc_path)

    t0_iso, t1_iso = timelims
    t0_cap, t1_cap = _cap_temporal_by_delay(t0_iso, t1_iso, delay_hours)
    bbox = _bbox(lonlims, latlims)

    # 搜索 granules
    results = earthaccess.search_data(
        short_name=COLLECTION_SHORT_NAME,
        temporal=(t0_cap, t1_cap),
        bounding_box=bbox,
        count=max_results if max_results else 2000,
    )

    rows = []
    for gran in results:
        links = [u.get("href") if isinstance(u, dict) else str(u) for u in gran.data_links()]
        https_links = [u for u in links if u.startswith("https://")]
        if not https_links:
            continue
        url = https_links[0]
        file = url.split("/")[-1]
        m = FILENAME_TIME_RE.search(file)
        ts = pd.NaT
        if m:
            ts = pd.to_datetime(m.group(1), format="%Y%m%d%H%M%S", utc=True)

        size_MB = None
        try:
            size_MB = float(gran.size()) / (1024 * 1024)
        except Exception:
            pass

        rows.append(
            dict(
                time_utc=ts,
                file=file,
                size_MB=size_MB,
                url=url,
                bbox=bbox,
            )
        )

    df = pd.DataFrame(rows).sort_values("time_utc").reset_index(drop=True)
    return df


In [7]:
df = query_himawari_l3c_manifest(
    timelims=("2025-03-01T00:00:00", "2025-03-02T00:00:00"),
    lonlims=(111, 116),
    latlims=(-24.5, -19.5),
    delay_hours=4,
    netrc_path=Path(".netrc"),
)

print(df.head())
df.to_csv("manifest.csv", index=False)


C:\Users\13217\AppData\Local\Temp\ipykernel_4416\75282727.py:30: UserWarning: no explicit representation of timezones available for np.datetime64
  now_safe = np.datetime64(pd.Timestamp.utcnow().to_pydatetime()) - np.timedelta64(delay_hours, "h")


                   time_utc  \
0 2025-03-01 00:00:00+00:00   
1 2025-03-01 01:00:00+00:00   
2 2025-03-01 02:00:00+00:00   
3 2025-03-01 03:00:00+00:00   
4 2025-03-01 04:00:00+00:00   

                                                file   size_MB  \
0  20250301000000-STAR-L3C_GHRSST-SSTsubskin-AHI_...  0.000040   
1  20250301010000-STAR-L3C_GHRSST-SSTsubskin-AHI_...  0.000042   
2  20250301020000-STAR-L3C_GHRSST-SSTsubskin-AHI_...  0.000042   
3  20250301030000-STAR-L3C_GHRSST-SSTsubskin-AHI_...  0.000044   
4  20250301040000-STAR-L3C_GHRSST-SSTsubskin-AHI_...  0.000044   

                                                 url                      bbox  
0  https://archive.podaac.earthdata.nasa.gov/poda...  (111, -24.5, 116, -19.5)  
1  https://archive.podaac.earthdata.nasa.gov/poda...  (111, -24.5, 116, -19.5)  
2  https://archive.podaac.earthdata.nasa.gov/poda...  (111, -24.5, 116, -19.5)  
3  https://archive.podaac.earthdata.nasa.gov/poda...  (111, -24.5, 116, -19.5)  
4  https://

In [8]:
def main():
    import os
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import xarray as xr
    import earthaccess
    import matplotlib.pyplot as plt

    # ==== 配置时间 & 区域 ====
    timelims = ("2025-03-01T00:00:00", "2025-03-01T12:00:00")
    tstep = 3600
    lonlims = (111, 116)
    latlims = (-24.5, -19.5)

    version = 9
    level = "L3C"   # L3C

    dtlims = (np.datetime64(timelims[0]), np.datetime64(timelims[1]))
    dtrange = np.arange(dtlims[0], dtlims[1], np.timedelta64(tstep, 's'))

    # ==== 数据集名称（Himawari-9 L3C）====
    short_name = "H09-AHI-L3C-ACSPO-v2.90"
    long_name  = "STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0"

    # 考虑 2-6 小时发布延迟，这里取 4 小时中值
    now_utc = np.datetime64(pd.Timestamp.utcnow().to_pydatetime())
    safe_latest = now_utc - np.timedelta64(4, 'h')
    dtrange = dtrange[dtrange <= safe_latest]

    print(f"Lon limits:  {lonlims}")
    print(f"Lat limits:  {latlims}")
    print(f"Time limits: {timelims} (capped to <= {pd.Timestamp(safe_latest)} UTC)")
    print(f"Version:     {version}, Level: {level}")

    # ==== EarthAccess 登录（使用 .netrc）====
    NETRC_FILE = (Path.cwd() / ".netrc").resolve()
    if not NETRC_FILE.exists():
        raise FileNotFoundError(
            f".netrc not found at {NETRC_FILE}\n"
            "machine urs.earthdata.nasa.gov\n  login YOUR_USERNAME\n  password YOUR_PASSWORD\n"
        )
    os.environ["NETRC"] = str(NETRC_FILE)
    for k in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
        os.environ.pop(k, None)
    auth = earthaccess.login(strategy="netrc", persist=True)
    if not auth.authenticated:
        raise RuntimeError(f"Earthdata login failed. Check {NETRC_FILE}")

    # ==== 目录 ====
    data_name = "himawari_l3c"
    base_dir = Path("data") / data_name
    temp_dir = base_dir / "temp"
    parts_dir = base_dir / "parts"
    png_dir = base_dir / "png"
    base_dir.mkdir(parents=True, exist_ok=True)
    temp_dir.mkdir(exist_ok=True)
    parts_dir.mkdir(exist_ok=True)
    png_dir.mkdir(exist_ok=True)

    # 小工具：找经纬度/时间维名称
    def pick_name(ds, candidates):
        for name in candidates:
            if name in ds.coords or name in ds.variables:
                return name
        raise KeyError(f"None of {candidates} found in dataset: {list(ds.coords) + list(ds.variables)}")

    for dt in dtrange:
        dt_pd = pd.Timestamp(dt)
        time_str = dt_pd.strftime("%Y%m%d%H%M%S")
        file_name = f"{time_str}-{long_name}.nc"
        link = f"https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/{short_name}/{file_name}"
        file_path = temp_dir / file_name
        output_path = parts_dir / f"{time_str}.nc"

        if output_path.exists():
            print(f"Already processed: {output_path}")
            continue

        # 下载
        if not file_path.exists():
            print(f"Downloading: {file_name}")
            paths = earthaccess.download(link, str(temp_dir))
            if not paths:
                print(f"Download failed or not available yet: {file_name}")
                continue
            file_on_disk = Path(paths[0])
        else:
            print(f"Using cached file: {file_path}")
            file_on_disk = file_path

        try:
            with xr.open_dataset(file_on_disk, engine="netcdf4") as ds:
                if level != "L3C":
                    raise ValueError("Expect level='L3C'")

                # 识别经纬度名，并保证纬度升序
                lon_name = pick_name(ds, ["lon", "longitude"])
                lat_name = pick_name(ds, ["lat", "latitude"])
                time_name = pick_name(ds, ["time", "t"])

                if np.asarray(ds[lat_name]).ndim == 1:
                    # 一维规则网格：确保升序，方便 slice(min,max)
                    if float(ds[lat_name].values[0]) > float(ds[lat_name].values[-1]):
                        ds = ds.sortby(lat_name)
                else:
                    # 罕见：二维经纬度网格（L3C 基本不会出现），这里简单跳过排序
                    pass

                lon_slice = slice(min(lonlims), max(lonlims))
                lat_slice = slice(min(latlims), max(latlims))
                ds_cropped = ds.sel({lon_name: lon_slice, lat_name: lat_slice})

                # 可能变量名差异，优先找 GHRSST 标准名
                var_candidates = ["sea_surface_temperature", "sst", "sea_surface_temperature_skin"]
                for v in var_candidates:
                    if v in ds_cropped.data_vars:
                        sst_name = v
                        break
                else:
                    raise KeyError(f"No SST var in {var_candidates}")

                # quality
                q_candidates = ["quality_level", "l2p_flags", "quality_level_sst"]
                q_name = None
                for q in q_candidates:
                    if q in ds_cropped.data_vars:
                        q_name = q
                        break

                keep_vars = [sst_name] + ([q_name] if q_name else [])
                ds_cropped = ds_cropped[keep_vars].copy()
                ds_cropped.attrs = {}

                # 若有多个 time 也无妨；一般只有一个
                ds_cropped.to_netcdf(output_path)
                print(f"Saved cropped dataset to {output_path}")

                # 画 PNG
                # 找第一个时间步（兼容 time/t 的不同名字）
                sst = ds_cropped[sst_name]
                if time_name in sst.dims and sst.sizes[time_name] > 1:
                    sst0 = sst.isel({time_name: 0})
                elif time_name in sst.dims and sst.sizes[time_name] == 1:
                    sst0 = sst.isel({time_name: 0})
                else:
                    sst0 = sst  # 无时间维

                # 处理全 NaN 的边缘情况
                vmin = float(np.nanmin(sst0.values)) if np.isfinite(sst0.values).any() else None
                vmax = float(np.nanmax(sst0.values)) if np.isfinite(sst0.values).any() else None
                if vmin is None or vmax is None or not np.isfinite([vmin, vmax]).all():
                    print(f"SST all-NaN for {time_str}, skip PNG.")
                else:
                    fig, ax = plt.subplots(figsize=(8, 6))
                    im = ax.imshow(sst0.values, origin="lower", cmap="turbo", vmin=vmin, vmax=vmax)
                    plt.colorbar(im, ax=ax, label="Sea Surface Temperature (K)")
                    ax.set_title(f"SST {time_str}")
                    plt.tight_layout()
                    png_path = (png_dir / f"{time_str}.png")
                    plt.savefig(png_path, dpi=150)
                    plt.close()
                    print(f"Saved PNG: {png_path}")

        finally:
            # 删除临时文件
            if file_on_disk.exists():
                try:
                    os.remove(file_on_disk)
                    print(f"Removed temp file: {file_on_disk}")
                except PermissionError:
                    print(f"Skipping removal, file still in use: {file_on_disk}")

if __name__ == "__main__":
    main()


C:\Users\13217\AppData\Local\Temp\ipykernel_4416\2499772493.py:27: UserWarning: no explicit representation of timezones available for np.datetime64
  now_utc = np.datetime64(pd.Timestamp.utcnow().to_pydatetime())


Lon limits:  (111, 116)
Lat limits:  (-24.5, -19.5)
Time limits: ('2025-03-01T00:00:00', '2025-03-01T12:00:00') (capped to <= 2025-08-31 01:53:35.194871 UTC)
Version:     9, Level: L3C
Downloading: 20250301000000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 1015.57it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:10<00:00, 10.76s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<?, ?it/s]


Saved cropped dataset to data\himawari_l3c\parts\20250301000000.nc
Saved PNG: data\himawari_l3c\png\20250301000000.png
Removed temp file: data\himawari_l3c\temp\20250301000000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc
Downloading: 20250301010000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<?, ?it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:10<00:00, 10.36s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<?, ?it/s]


...

Saved cropped dataset to data\himawari_l3c\parts\20250301090000.nc
Saved PNG: data\himawari_l3c\png\20250301090000.png
Removed temp file: data\himawari_l3c\temp\20250301090000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc
Downloading: 20250301100000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<?, ?it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:11<00:00, 11.66s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<?, ?it/s]


Saved cropped dataset to data\himawari_l3c\parts\20250301100000.nc
Saved PNG: data\himawari_l3c\png\20250301100000.png
Removed temp file: data\himawari_l3c\temp\20250301100000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc
Downloading: 20250301110000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<?, ?it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:10<00:00, 10.68s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<?, ?it/s]


Saved cropped dataset to data\himawari_l3c\parts\20250301110000.nc
Saved PNG: data\himawari_l3c\png\20250301110000.png
Removed temp file: data\himawari_l3c\temp\20250301110000-STAR-L3C_GHRSST-SSTsubskin-AHI_H09-ACSPO_V2.90-v02.0-fv01.0.nc


In [10]:
import xarray as xr
from pathlib import Path
import pandas as pd

def merge_parts_to_single_nc(parts_dir: Path, merged_path: Path):
    """
    合并 parts/ 目录下的所有裁剪后的 nc 文件为一个文件
    """
    # 找到所有裁剪后的 nc 文件
    nc_files = sorted(parts_dir.glob("*.nc"))
    if not nc_files:
        print(f"[WARN] No files found in {parts_dir}")
        return
    
    print(f"Found {len(nc_files)} files, merging...")

    # 读取所有 nc 文件
    datasets = []
    for f in nc_files:
        ds = xr.open_dataset(f)
        # 确保 time 是 datetime64
        if 'time' in ds:
            ds['time'] = pd.to_datetime(ds['time'].values)
        datasets.append(ds)
    
    # 按时间维度拼接
    merged_ds = xr.concat(datasets, dim="time")
    
    # 保存为一个新的大 nc
    merged_ds.to_netcdf(merged_path)
    print(f"[OK] Merged dataset saved to {merged_path}")

    # 关闭所有小文件，释放内存
    for ds in datasets:
        ds.close()
    merged_ds.close()

# 使用方法
parts_dir = Path("data/himawari_l3c/parts")
merged_path = Path("data/himawari_l3c/merged_sst.nc")
merge_parts_to_single_nc(parts_dir, merged_path)



Found 12 files, merging...
[OK] Merged dataset saved to data\himawari_l3c\merged_sst.nc


In [13]:
import xarray as xr

# 读取合并后的文件
ds = xr.open_dataset("merged_sst.nc")

# 查看所有变量
print(ds.data_vars)

# 获取海表温度数据
sst = ds["sea_surface_temperature"]
print(sst.shape)   # (12, 250, 250)

# 获取时间坐标
print(ds["time"].values)



Data variables:
    sea_surface_temperature  (time, lat, lon) float32 3MB ...
    quality_level            (time, lat, lon) float32 3MB ...
(12, 250, 250)
['2025-03-01T00:00:00.000000000' '2025-03-01T01:00:00.000000000'
 '2025-03-01T02:00:00.000000000' '2025-03-01T03:00:00.000000000'
 '2025-03-01T04:00:00.000000000' '2025-03-01T05:00:00.000000000'
 '2025-03-01T06:00:00.000000000' '2025-03-01T07:00:00.000000000'
 '2025-03-01T08:00:00.000000000' '2025-03-01T09:00:00.000000000'
 '2025-03-01T10:00:00.000000000' '2025-03-01T11:00:00.000000000']
